# Task 2 — Incremental CPG Parser Service

## Mục tiêu

Xử lý từng file trong bounded memory, trích AST, CFG, DFG và call edge, gán ID
ổn định rồi phát event theo hợp đồng chung.

## Thiết kế và triển khai

[`task2/cpg_parser.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task2/cpg_parser.py) dùng `ast`, structural path
và SHA-256. [`task2/parser_service.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task2/parser_service.py) đọc
manifest theo dòng, chỉ giữ một file trong bộ nhớ, phát UPSERT/DELETE và lưu
state sau khi Kafka xác nhận batch. Contract thống nhất các trường
`schema_version`, `path`, `content_sha256`, `source_id`, `target_id` và edge
type `CALL`.

Dry-run kiểm tra parser trên toàn bộ 147 file và tạo JSONL cục bộ. Kafka mode
phát cùng corpus vào bốn topic đang chạy; hai chế độ có summary riêng.

## Lệnh thực thi

```bash
python task2/parser_service.py --manifest artifacts/task1/python_manifest.jsonl \
  --repo-dir .work/repos/datasets --dry-run --output-dir artifacts/task2

python task2/parser_service.py --manifest artifacts/task1/python_manifest.jsonl \
  --repo-dir .work/repos/datasets --kafka-bootstrap "$KAFKA_BOOTSTRAP" \
  --summary-output artifacts/task2/kafka_publish_summary.json
```

## Kết quả thực nghiệm

Cell sau đọc cả dry-run summary, Kafka publish summary và corpus verifier.


In [1]:
import json
from pathlib import Path

def read(name):
    return json.loads(Path(name).read_text(encoding="utf-8"))

dry = read("../artifacts/task2/summary.json")
live = read("../artifacts/task2/kafka_publish_summary.json")
corpus = read("../artifacts/task2/corpus_verification.json")
schema = read("../artifacts/task2/schema_validation.json")

assert dry["dry_run"] is True and live["dry_run"] is False
for result in (dry, live):
    assert result["processed_files"] == 147
    assert result["error_files"] == 0
    assert result["total_nodes_emitted"] == 208003
    assert result["total_edges_emitted"] == 267695
assert corpus["distinct_node_ids"] == 208003
assert corpus["distinct_edge_ids"] == 267695
assert corpus["missing_edge_sources"] == 0
assert corpus["missing_edge_targets"] == 0

print("mode     files errors nodes  edges  duration_sec")
for mode, result in (("dry-run", dry), ("Kafka", live)):
    print(
        f"{mode:<8} {result['processed_files']:>5} {result['error_files']:>6} "
        f"{result['total_nodes_emitted']:>6} {result['total_edges_emitted']:>6} "
        f"{result['execution_duration_sec']:>12}"
    )
print("\ncorpus:", json.dumps(corpus, indent=2))
print("schema:", json.dumps(schema, indent=2))


mode     files errors nodes  edges  duration_sec
dry-run    147      0 208003 267695      326.198
Kafka      147      0 208003 267695       617.13

corpus: {
  "node_events": 208003,
  "distinct_node_ids": 208003,
  "edge_events": 267695,
  "distinct_edge_ids": 267695,
  "metadata_events": 147,
  "distinct_metadata_file_ids": 147,
  "error_events": 0,
  "missing_edge_sources": 0,
  "missing_edge_targets": 0,
  "status": "PASS"
}
schema: {
  "status": "PASS",
  "command": "python -m unittest tests.test_event_contract -v",
  "source": "Task 2 publish_success, replay DELETE, and publish_failure events",
  "schemas": [
    "node-event.schema.json",
    "edge-event.schema.json",
    "metadata-event.schema.json",
    "error-event.schema.json"
  ],
  "tests": 2
}


## Vấn đề và cách xử lý

Contract cũ từng lệch giữa `CALLS`/`CALL`, `file_path`/`path` và kiểu
`schema_version`; các biến thể được thay bằng một contract và JSON Schema duy
nhất. Lần publish đầu dừng ở file lớn vì ACK timeout và Docker thiếu RAM. Producer
được đổi sang batch 1.000 event, gzip và timeout dài hơn; Compose giới hạn heap
cho Kafka, Kafka Connect, Neo4j và Spark. Lần chạy lại hoàn thành 147 file.

## Đánh giá

Cả dry-run và Kafka mode đều cho 208.003 node, 267.695 edge, 0 parser error và
không thiếu endpoint. Giới hạn còn lại là thời gian publish phụ thuộc tài nguyên
Docker; summary có `fatal_error` nếu hạ tầng dừng giữa chừng.
